In [9]:
# gds.ipynb — экспорт GDS для фабрикации

%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

from pcm_pix.run import start_run
from pcm_pix.solution import load_solution, Solution
from pcm_pix.gds import GDSFabCfg, export_fabrication_gds

CFG = {
    "run_name": "PCM_bagel_2025",
    "data_dir": "data",
    "outputs_dir": "outputs",
    "preset_name": "pos_server_2026_03_29",
    "preset_dir": "./data/preset",
    "Nn": 11,
    "b_min_m": 50e-9,
    "gds_l_m": 10e-6,
    "gds_L_m": 11 * 10e-6,
    # layer: True → обе ячейки layer=0; False → layer += 1 каждые gds_layer_step_n дисков
    "gds_layer_all_zero": False,
    "gds_layer_step_n": 1000,
    # второй cell: Al2O3 (внешний радиус +r, внутренний полурадиус −r)
    "gds_al2o3_cover": True,
    "gds_al2o3_r_m": 100e-9,
    "gds_al2o3_cell_name": "CELL_1",
}

run = start_run(outputs_dir=CFG["outputs_dir"], run_name=CFG["run_name"])


2026-03-29 18:45:37,610 | INFO | run_dir=/media/slava/Data/git/PCM_pix/outputs/PCM_bagel_2025


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# --- Load pos from preset ---
# Можно запускать независимо от порядка ячеек: берём CFG из globals() + дефолты.
CFG = globals().get("CFG") or {}

preset_name = str(CFG.get("preset_name", "pos_server_2026_03_29"))
preset_dir = Path(str(CFG.get("preset_dir", "./data/preset"))).expanduser()

sol_path = preset_dir / f"{preset_name}.json"
if not sol_path.exists():
    raise FileNotFoundError(f"Preset not found: {sol_path}")

In [11]:
pos, cost, meta = load_solution(sol_path)

Nn = int(CFG.get("Nn", 11))
b_min_m = float(CFG.get("b_min_m", 50e-9))

a_m, d_m, b_m = Solution.split_to_adb(pos, Nn=Nn, b_min_m=b_min_m)

a_m, d_m, b_m, float(cost) if cost is not None else None

(array([9.99630887e-07, 9.18393530e-07, 8.44642448e-07, 8.98831257e-07,
        9.41116076e-07, 8.19523707e-07, 8.24034874e-07, 7.91565078e-07,
        8.06523175e-07, 8.07848937e-07, 9.97401059e-07]),
 array([6.44568646e-07, 6.82222323e-07, 7.08280597e-07, 7.44097866e-07,
        6.50580775e-07, 5.05616500e-07, 5.11047216e-07, 5.54870203e-07,
        5.51678127e-07, 1.61628112e-07, 7.35584785e-07]),
 array([3.12574887e-07, 2.84143361e-07, 2.85582671e-07, 4.11824665e-07,
        3.35166722e-07, 5.15943744e-08, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 3.46320431e-07]),
 None)

In [12]:
# --- Export GDS (fabrication) ---
# Параметры как в 3rd art: l, L; опционально CELL_1 (Al2O3) через gds_al2o3_*
cfg_gds = GDSFabCfg(
    name=str(CFG.get("preset_name")),
    l_m=float(CFG.get("gds_l_m", 20e-6)),
    L_m=float(CFG.get("gds_L_m", 11 * 20e-6)),
    layer_all_zero=bool(CFG.get("gds_layer_all_zero", True)),
    layer_step_n=int(CFG.get("gds_layer_step_n", 1000)),
    al2o3_cover_enable=bool(CFG.get("gds_al2o3_cover", False)),
    al2o3_r_m=float(CFG.get("gds_al2o3_r_m", 100e-9)),
    al2o3_cell_name=str(CFG.get("gds_al2o3_cell_name", "CELL_1")),
)

meta_out = {
    "preset_name": preset_name,
    "cost": cost,
    "source_meta": meta,
}

# Пишем в outputs/<run>/gds/
gds_path, txt_path = export_fabrication_gds(
    a_m=a_m,
    d_m=d_m,
    b_m=b_m,
    out_dir=run.gds,
    cfg=cfg_gds,
    meta=meta_out,
)

gds_path, txt_path

(PosixPath('outputs/PCM_bagel_2025/gds/pos_server_2026_03_29.gds'),
 PosixPath('outputs/PCM_bagel_2025/gds/pos_server_2026_03_29.txt'))